In [ ]:
import os
import math
import random
from typing import Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision import models

import matplotlib.pyplot as plt

# Reproducibility settings
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# TODO set random tran test val split later

DATA_DIR = "Data"

TRAIN_DIR = os.path.join(DATA_DIR, "Train")
VAL_DIR   = os.path.join(DATA_DIR, "Val")

assert os.path.isdir(TRAIN_DIR), f"Train directory not found: {TRAIN_DIR}"
assert os.path.isdir(VAL_DIR), f"Val directory not found: {VAL_DIR}"

print("Train dir:", TRAIN_DIR)
print("Val dir:", VAL_DIR)


In [ ]:
IMG_SIZE = 224

# ImageNet normalization stats (used by pretrained models)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def to_3ch(x: torch.Tensor) -> torch.Tensor:
    # x is shape (1,H,W) after Grayscale+ToTensor.
    # Repeat to (3,H,W) for ImageNet-pretrained CNNs.
    return x.repeat(3, 1, 1)

train_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=7),
    transforms.ToTensor(),
    transforms.Lambda(to_3ch),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Lambda(to_3ch),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


In [ ]:
train_ds = ImageFolder(TRAIN_DIR, transform=train_tfms)
val_ds   = ImageFolder(VAL_DIR,   transform=val_tfms)

print("Class-to-index mapping:", train_ds.class_to_idx)
class_names = train_ds.classes

BATCH_SIZE = 32  # ResNet is reasonably light; adjust batch size for your hardware
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print("Batch x shape:", xb.shape, "Batch y shape:", yb.shape)

In [ ]:
def get_resnet18_pretrained(num_classes: int):
    try:
        # Newer torchvision
        weights = models.ResNet18_Weights.DEFAULT
        model = models.resnet18(weights=weights)
    except Exception:
        # Older torchvision
        model = models.resnet18(pretrained=True)

    # Replace final layer: `fc` is the classifier in ResNet
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

model = get_resnet18_pretrained(num_classes=len(class_names)).to(device)
model

In [ ]:
def set_trainable(module: nn.Module, trainable: bool):
    for p in module.parameters():
        p.requires_grad = trainable

# Freeze everything
set_trainable(model, False)

# Unfreeze only the head
set_trainable(model.fc, True)

# Sanity check: count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,} / {all_params:,} ({100*trainable_params/all_params:.2f}%)")

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion) -> Tuple[float, float, np.ndarray, np.ndarray]:
    # Returns: avg_loss, accuracy, y_true (N,), y_prob (N,2)
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    y_true_all = []
    y_prob_all = []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == y).sum().item()
        total += x.size(0)

        probs = torch.softmax(logits, dim=1)  # (B,2)
        y_true_all.append(y.detach().cpu().numpy())
        y_prob_all.append(probs.detach().cpu().numpy())

    y_true_all = np.concatenate(y_true_all, axis=0)
    y_prob_all = np.concatenate(y_prob_all, axis=0)
    return total_loss / total, total_correct / total, y_true_all, y_prob_all

def train_model (model, train_loader, val_loader, epochs=10, lr=1e-3, weight_decay=1e-4):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad],
                           lr=lr, weight_decay=weight_decay)    # Training loop
    
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):  # Run for 10 epochs
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            running_correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
        
        train_loss = running_loss / total
        train_acc = running_correct / total
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch:02d}/{epochs} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}")
        
    return history

def plot_curves(history: Dict[str, list]):
    plt.figure(figsize=(7,4))
    plt.plot(history["train_loss"], label="train_loss")
    plt.plot(history["val_loss"], label="val_loss")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("Loss curves")
    plt.show()

    plt.figure(figsize=(7,4))
    plt.plot(history["train_acc"], label="train_acc")
    plt.plot(history["val_acc"], label="val_acc")
    plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.title("Accuracy curves")
    plt.show()

In [ ]:
# TODO play around with freezing layers
history_head = train_model(model, train_loader, val_loader, epochs=5, lr=1e-3, weight_decay=1e-4)
plot_curves(history_head)